In [1]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import optuna
from tqdm import tqdm
import numpy as np

# load data

In [2]:
# load training dataset
train_path = "data/basic_dataset.csv"
chunk = pd.read_csv(train_path, chunksize=1000000)
df = pd.concat(chunk)

# data preparation

In [3]:
# split data based on search ids
train_groups, val_groups = train_test_split(df[['srch_id']].drop_duplicates(), test_size=0.1, random_state=42)

# split the actual data on those groups
train_data = df[df['srch_id'].isin(train_groups['srch_id'])]
val_data = df[df['srch_id'].isin(val_groups['srch_id'])]

# prepare X and y for train/val set
X_train = train_data.drop(columns=['click_bool', 'booking_bool', 'relevance', 'position', 'gross_bookings_usd'])
y_train = train_data['relevance']

X_val = val_data.drop(columns=['click_bool', 'booking_bool', 'relevance', 'position', 'gross_bookings_usd'])
y_val = val_data['relevance']

In [4]:
# create groups (for DMatrix)
train_group = train_data.groupby('srch_id').size().to_list()
val_group = val_data.groupby('srch_id').size().to_list()

dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_group(train_group)

dval = xgb.DMatrix(X_val, label=y_val)
dval.set_group(val_group)

# Train model

In [5]:
params = {
    'objective': 'rank:pairwise',
    'eta': 0.1, # learning rate
    'gamma': 0.5, # minimum loss reduction required to further partition on a leaf node 
    'min_child_weight': 0.1, #  minimum sum of instance weights needed in a child
    'max_depth': 10, # max depth of the tree
    'eval_metric': 'ndcg' # 'map' for mean average precision; 'ndcg' for ndcg
}

In [6]:
# Define the objective function
def objective(trial):
    param = {
        'objective': 'rank:ndcg',
        'eval_metric': 'ndcg',
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        #'n_estimators': trial.suggest_int('n_estimators', 100, 1000), # not used
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    
    # load data
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(train_group)
    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(val_group)
    
    result = xgb.cv(param, dtrain, num_boost_round=250, nfold=5,
                    early_stopping_rounds=10, metrics='ndcg', seed=420)
    
    return result['test-ndcg-mean'].max()

In [7]:
n_trials = 10

# progressbar
pbar = tqdm(total=n_trials, desc="Optimization Progress")
def tqdm_callback(study, trial):
    pbar.update(1)

# Optimize the hyperparameters
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=n_trials, callbacks=[tqdm_callback])

pbar.close()

print("Best hyperparameters: ", study.best_params)

Optimization Progress:   0%|          | 0/10 [00:00<?, ?it/s][I 2024-05-18 18:42:46,734] A new study created in memory with name: no-name-ccf392e4-13a3-4199-8a92-a1a4863ac737


[18:43:02] WARNING: C:\b\abs_7diruzi3as\croot\xgboost-split_1712794727514\work\src\learner.cc:767: 
Parameters: { "n_estimators" } are not used.
[18:43:07] WARNING: C:\b\abs_7diruzi3as\croot\xgboost-split_1712794727514\work\src\learner.cc:767: 
Parameters: { "n_estimators" } are not used.
[18:43:12] WARNING: C:\b\abs_7diruzi3as\croot\xgboost-split_1712794727514\work\src\learner.cc:767: 
Parameters: { "n_estimators" } are not used.
[18:43:17] WARNING: C:\b\abs_7diruzi3as\croot\xgboost-split_1712794727514\work\src\learner.cc:767: 
Parameters: { "n_estimators" } are not used.
[18:43:22] WARNING: C:\b\abs_7diruzi3as\croot\xgboost-split_1712794727514\work\src\learner.cc:767: 
Parameters: { "n_estimators" } are not used.


[W 2024-05-18 18:47:23,067] Trial 0 failed with parameters: {'lambda': 0.42581195973331587, 'alpha': 0.01753613983213082, 'colsample_bytree': 0.7557039967133966, 'subsample': 0.8580812532435497, 'learning_rate': 0.02591148817490586, 'max_depth': 7, 'n_estimators': 805, 'min_child_weight': 9} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\Dany\anaconda3\envs\ml24\lib\site-packages\optuna\study\_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Dany\AppData\Local\Temp\ipykernel_10316\963306867.py", line 23, in objective
    result = xgb.cv(param, dtrain, num_boost_round=250, nfold=5,
  File "C:\Users\Dany\anaconda3\envs\ml24\lib\site-packages\xgboost\training.py", line 538, in cv
    booster.update(i, obj)
  File "C:\Users\Dany\anaconda3\envs\ml24\lib\site-packages\xgboost\training.py", line 229, in update
    fold.update(iteration, obj)
  File "C:\Users\Dany\anaconda3\envs\ml24\lib\site-pack

KeyboardInterrupt: 

In [ ]:
# save best model to save_models/

# test best model
evals_result = {} # save results into dict
bst = xgb.train(dtrain, epochs=300, params=study.best_params, evals=[(dtrain, 'train'), (dval, 'val')], early_stopping_rounds=10, evals_result=evals_result, verbose_eval=True)

best_iteration = bst.best_iteration
best_score = bst.best_score
bst.save_model(f'save_models/best_model_{best_iteration}_score_{best_score:.2f}.model')

In [ ]:
# plot training
train_evals = evals_result['train']['ndcg']
val_evals = evals_result['val']['ndcg']
plt.figure(figsize=(10, 5))
plt.plot(train_evals, label='Train NDCG')
plt.plot(val_evals, label='Validation NDCG')
plt.xlabel('Iterations')
plt.ylabel('NDCG')
plt.title('Training Vs. Validation NDCG')
plt.legend()
plt.ylim()
plt.show()

In [ ]:
# validation set
preds = bst.predict(dval)
print(preds)

Run model on the test_data:

In [ ]:
# load test set
test_path = "data/test_set_VU_DM.csv"
chunk = pd.read_csv(test_path, chunksize=1000000)
test_df = pd.concat(chunk)

# copied from data_prep.ipynb
test_df = test_df.drop(['date_time','orig_destination_distance','comp1_rate','comp1_inv','comp1_rate_percent_diff','comp2_rate','comp2_inv','comp2_rate_percent_diff','comp3_rate','comp3_inv','comp3_rate_percent_diff','comp4_rate','comp4_inv','comp4_rate_percent_diff','comp5_rate','comp5_inv','comp5_rate_percent_diff','comp6_rate','comp6_inv','comp6_rate_percent_diff','comp7_rate','comp7_inv','comp7_rate_percent_diff','comp8_rate','comp8_inv','comp8_rate_percent_diff','srch_query_affinity_score', 'visitor_hist_starrating','visitor_hist_adr_usd'],axis=1)

In [ ]:
dtest = xgb.DMatrix(test_df)

In [ ]:
test_preds = bst.predict(dtest)
print(test_preds)

In [ ]:
# add predictions as column
test_df['prediction'] = test_preds

grouped = test_df.groupby('srch_id').apply(lambda x: x.sort_values('prediction', ascending=False))

# Reset the index to flatten the DataFrame
grouped = grouped.reset_index(drop=True)

# Select the top 5 for each search_id
top5 = grouped.groupby('srch_id')

print(top5)

In [ ]:
grouped[['srch_id', 'prop_id']].to_csv('kaggle_predictions/xgboost_predictions.csv', index=False, sep=',')